# Estudo dirigido sobre agentes

## O Ambiente

O **Vacuum World**, proposto por Russell & Norvig em *Artificial Intelligence: A Modern Approach*, é um ambiente de grade bidimensional composto por células que podem estar **limpas**, **sujas** ou ocupadas por **obstáculos**. Um agente autônomo navega por esse espaço com o objetivo de limpar o maior número de células possível dentro de um número finito de passos.

## PEAS

| Componente | Descrição |
|---|---|
| **Performance** | Pontuação acumulada com base na limpeza e no custo de movimentação (definido na próxima célula) |
| **Environment** | Grade $N \times M$ com sujeira e obstáculos distribuídos aleatoriamente; parcialmente observável |
| **Actuators** | Mover (↑ ↓ ← →), Aspirar |
| **Sensors** | Estado da célula atual (suja / limpa / obstáculo) e células adjacentes imediatas |

## Hipótese

Agentes dotados de memória interna produzirão trajetórias mais eficientes do que agentes puramente reativos, resultando em desempenho superior sob métricas que penalizam movimentos desnecessários.

## Métricas de Avaliação

Dois critérios de desempenho são adotados para avaliar os agentes. Ambos recompensam a limpeza, mas divergem na forma como tratam o custo de movimentação.

---

### Métrica 1 — Cobertura Pura

$$M_1 = \sum_{t=1}^{T} \mathbb{1}[\text{célula } s_t \text{ foi limpa no passo } t]$$

Cada célula limpa pelo agente contribui com **+1 ponto**. Movimentos não têm custo. Esta métrica favorece agentes que maximizam cobertura independentemente da trajetória percorrida — um agente que se move em círculos ainda pode pontuar bem, desde que eventualmente passe por células sujas.

---

### Métrica 2 — Eficiência (Cobertura com Penalidade)

$$M_2 = \sum_{t=1}^{T} \mathbb{1}[\text{célula } s_t \text{ foi limpa no passo } t] - \sum_{t=1}^{T} \mathbb{1}[\text{agente se moveu no passo } t]$$

Cada célula limpa contribui com **+1 ponto**; cada movimento executado desconta **−1 ponto**. Esta métrica penaliza trajetórias redundantes — um agente que revisita células já limpas ou colide repetidamente com obstáculos acumula custo de movimento sem contrapartida de pontuação, tornando a **eficiência de caminho** um fator determinante.

In [ ]:
import numpy as np
import random
from typing import Optional

CLEAN = 0
DIRTY = 1
OBSTACLE = 2

ACTIONS = ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SUCK', 'IDLE']

DELTA = {
    'UP': (-1, 0),
    'DOWN': (1, 0),
    'LEFT': (0, -1),
    'RIGHT': (0, 1),
}


class Environment:
    """
    Ambiente de grade parcialmente observável para o agente aspirador.

    A grade utiliza três estados: LIMPO (0), SUJO (1) e OBSTÁCULO (2).
    As bordas do ambiente são sempre preenchidas por obstáculos. A observação
    do agente é local, restrita à sua célula atual e aos quatro vizinhos cardeais.
    """

    def __init__(
        self,
        rows: int = 8,
        cols: int = 8,
        dirt_prob: float = 0.3,
        obstacle_prob: float = 0.15,
        seed: Optional[int] = None,
    ):
        """
        Inicializa o ambiente de simulação.

        Args:
            rows (int): Número de linhas da grade.
            cols (int): Número de colunas da grade.
            dirt_prob (float): Probabilidade de geração de sujeira por célula.
            obstacle_prob (float): Probabilidade de geração de obstáculo por célula.
            seed (Optional[int]): Semente do gerador aleatório para reprodutibilidade.
        """
        self.rows = rows
        self.cols = cols
        self.dirt_prob = dirt_prob
        self.obstacle_prob = obstacle_prob
        self.seed = seed
        self.last_bump = False
        self.grid: np.ndarray = np.zeros((rows, cols), dtype=int)
        self.agent_pos: tuple[int, int] = (1, 1)
        self.initial_dirty: int = 0
        self._build()


    def _build(self) -> None:
        """
        Constrói a matriz do ambiente.
        Aplica obstáculos nas bordas, distribui sujeira e obstáculos internos com base
        nas probabilidades e inicializa a posição do agente em (1, 1).
        """
        rng = random.Random(self.seed)
        grid = np.zeros((self.rows, self.cols), dtype=int)
        grid[0, :] = OBSTACLE
        grid[-1, :] = OBSTACLE
        grid[:, 0] = OBSTACLE
        grid[:, -1] = OBSTACLE

        for r in range(1, self.rows - 1):
            for c in range(1, self.cols - 1):
                roll = rng.random()
                if roll < self.obstacle_prob:
                    grid[r, c] = OBSTACLE
                elif roll < self.obstacle_prob + self.dirt_prob:
                    grid[r, c] = DIRTY

        spawn = (1, 1)
        grid[spawn] = CLEAN

        self.grid = grid
        self.agent_pos = spawn
        self.initial_dirty = int(np.sum(self.grid == DIRTY))

    def get_percept(self) -> dict:
      r, c = self.agent_pos
      return {
          'dirty': self.grid[r, c] == DIRTY,
          'bump':  self.last_bump,
      }


    def apply_action(self, action: str) -> None:
      self.last_bump = False
      if action == 'SUCK':
          self.grid[self.agent_pos] = CLEAN
          return
      if action == 'IDLE':
          return
      if action in DELTA:
          dr, dc = DELTA[action]
          nr, nc = self.agent_pos[0] + dr, self.agent_pos[1] + dc
          if self.grid[nr, nc] == OBSTACLE:
              self.last_bump = True
          else:
              self.agent_pos = (nr, nc)
          return
      raise ValueError(f"Unknown action: '{action}'")


    def is_done(self) -> bool:
        """
        Verifica a condição de término do episódio.

        Returns:
            bool: True se não houver mais células com estado SUJO (1), False caso contrário.
        """
        return int(np.sum(self.grid == DIRTY)) == 0

    def __repr__(self) -> str:
        symbols = {CLEAN: '·', DIRTY: '*', OBSTACLE: '█'}
        rows = []
        for r in range(self.rows):
            row = ''
            for c in range(self.cols):
                if (r, c) == self.agent_pos:
                    row += 'A '
                else:
                    row += symbols[self.grid[r, c]] + ' '
            rows.append(row)
        return '\n'.join(rows)

## Os Contendores

Dois agentes serão submetidos ao mesmo ambiente. Ambos os agentes recebem apenas dois bits de informação:

- `dirty` — a célula atual está suja?
- `bump` — o último movimento resultou em colisão com um obstáculo?

Nenhum agente conhece sua posição absoluta, o layout do mapa ou o estado de células vizinhas.

---

### Agente Reativo Simples

Este agente não possui memória. Cada decisão é tomada exclusivamente com base no percepto do instante atual, sem qualquer registro do passado.

Sua lógica resume-se a duas regras:

- Se `dirty == True` → **aspirar**
- Se `bump == True` → **girar** (sortear nova direção aleatoriamente)
- Caso contrário → **avançar** na direção atual

Como não há memória, o agente é incapaz de saber se já visitou uma célula.

---

### Agente Baseado em Modelos

Este agente mantém um **estado interno** construído incrementalmente a partir do histórico de perceptos e ações. Ele infere a sua posição relativa e constrói um mapa interno das células visitadas, limpas e bloqueadas.

A sua estratégia de navegação é orientada por **Busca em Largura (BFS)**:

- O mapa interno é consultado para identificar a célula suja ou inexplorada mais próxima
- BFS computa o caminho mínimo até essa célula, evitando posições já identificadas como obstáculo
- O agente executa esse caminho passo a passo, refinando o modelo a cada novo percepto

> A restrição percetual é idêntica para ambos os agentes. O que os separa é a capacidade de **acumular e raciocinar sobre o passado**.

In [ ]:
import random
from collections import deque
from abc import ABC, abstractmethod

UNKNOWN = -1

class Agent(ABC):
    """Classe base abstrata para todos os agentes de simulação."""

    def __init__(self):
        """Inicializa as propriedades base do agente."""
        self.actions_taken: list[str] = []

    def record(self, action: str) -> None:
        """
        Registra a ação executada no histórico do agente.

        Args:
            action (str): Ação realizada.
        """
        self.actions_taken.append(action)

    @abstractmethod
    def decide(self, percept: dict) -> str:
        """
        Determina a próxima ação com base na percepção local e/ou estado interno.

        Args:
            percept (dict): Percepção fornecida pelo ambiente.

        Returns:
            str: Ação escolhida ('UP', 'DOWN', 'LEFT', 'RIGHT', 'SUCK', 'IDLE').
        """
        ...

    def reset(self) -> None:
        """Redefine o histórico de ações do agente."""
        self.actions_taken = []


class SimpleReactiveAgent(Agent):
    """
    Agente reativo simples (sem estado interno).

    Política de decisão:
        1. Célula suja -> Aspira ('SUCK').
        2. Colisão detectada ('bump') -> Sorteia nova direção aleatória.
        3. Caso contrário -> Avança na direção atual.
    """

    MOVE_ACTIONS = ['UP', 'DOWN', 'LEFT', 'RIGHT']

    def __init__(self, seed: int | None = None):
        """
        Inicializa o agente reativo.

        Args:
            seed (int | None): Semente aleatória para garantir reprodutibilidade.
        """
        super().__init__()
        self._rng       = random.Random(seed)
        self._direction = self._rng.choice(self.MOVE_ACTIONS)

    def decide(self, percept: dict) -> str:
        """Aplica a política reativa com base na percepção instantânea."""
        if percept['dirty']:
            return 'SUCK'
        if percept['bump']:
            self._direction = self._rng.choice(self.MOVE_ACTIONS)
        return self._direction

    def reset(self) -> None:
        """Redefine o histórico e sorteia uma nova direção inicial."""
        super().reset()
        self._direction = self._rng.choice(self.MOVE_ACTIONS)



_MOVE_DELTA = {
    'UP': (-1, 0),
    'DOWN': (1, 0),
    'LEFT': (0, -1),
    'RIGHT': (0, 1),
}


class ModelBasedAgent(Agent):
    """
    Agente baseado em modelo que utiliza mapeamento interno e planejamento de rotas via busca em largura (BFS).
    """

    def __init__(self):
        """
        Inicializa o estado mental do agente, definindo a posição inicial (0,0),
        o mapa rastreado, a fila do plano atual e o cache da última ação.
        """
        super().__init__()
        self._pos: tuple[int, int] = (0, 0)
        self._map: dict[tuple[int, int], int] = {(0, 0): CLEAN}
        self._path: list[str] = []
        self._last_action: str | None = None


    def decide(self, percept: dict) -> str:
        """
        Ciclo principal de decisão guiado pelo modelo interno.

        Ordem de prioridade:
            1. Limpar (SUCK) se a célula atual estiver suja.
            2. Executar a próxima ação do plano de rota (BFS) existente.
            3. Calcular um novo plano em direção à sujeira ou área não explorada mais próxima.
        """

        self._update_model(percept)
        if percept['dirty']:
            self._map[self._pos] = DIRTY
            return self._dispatch('SUCK')
        if self._path:
            return self._dispatch(self._path.pop(0))
        target = self._bfs_target()
        if target is None:
            return self._dispatch('IDLE')
        self._path = self._bfs_path(self._pos, target)
        if not self._path:
            self._map[target] = OBSTACLE
            return self._dispatch('IDLE')

        return self._dispatch(self._path.pop(0))

    def _dispatch(self, action: str) -> str:
        """
        Salva a ação como 'última executada' para validação de movimento no próximo ciclo.

        Args:
            action (str): Ação decidida.

        Returns:
            str: A mesma ação (para encadeamento de retornos).
        """
        self._last_action = action
        return action

    def _update_model(self, percept: dict) -> None:
        """
        Sincroniza o mapa interno com o resultado da última ação e a percepção atual.

        Infere colisões para mapear obstáculos, atualiza as coordenadas da posição
        e registra células vizinhas não visitadas como estado DESCONHECIDO.

        Args:
            percept (dict): Leitura sensorial atual fornecida pelo ambiente.
        """
        if self._last_action in _MOVE_DELTA:
            dr, dc = _MOVE_DELTA[self._last_action]
            intended = (self._pos[0] + dr, self._pos[1] + dc)

            if percept['bump']:
                self._map[intended] = OBSTACLE
                self._path = []
            else:
                self._pos = intended

        self._map[self._pos] = DIRTY if percept['dirty'] else CLEAN
        for dr, dc in _MOVE_DELTA.values():
            nb = (self._pos[0] + dr, self._pos[1] + dc)
            if nb not in self._map:
                self._map[nb] = UNKNOWN


    def _bfs_target(self) -> tuple[int, int] | None:
        """
        Localiza o destino ótimo mais próximo através de busca em largura (BFS).

        Returns:
            tuple[int, int] | None: Coordenadas da célula SUJA mais próxima. Caso não haja
            sujeira conhecida, retorna a célula DESCONHECIDA mais próxima. Retorna None
            se o mapa estiver inteiramente limpo e explorado.
        """

        visited       = {self._pos}
        queue         = deque([self._pos])
        dirty_target  = None
        unknown_target= None

        while queue:
            current = queue.popleft()
            state   = self._map.get(current, UNKNOWN)

            if current != self._pos:
                if state == DIRTY and dirty_target is None:
                    dirty_target = current
                    break
                if state == UNKNOWN and unknown_target is None:
                    unknown_target = current

            for dr, dc in _MOVE_DELTA.values():
                nb = (current[0] + dr, current[1] + dc)
                if nb not in visited and self._map.get(nb, UNKNOWN) != OBSTACLE:
                    visited.add(nb)
                    queue.append(nb)

        return dirty_target or unknown_target

    def _bfs_path(
        self,
        start: tuple[int, int],
        goal:  tuple[int, int],
    ) -> list[str]:
        """
        Gera a rota mais curta entre dois pontos do mapa. Assume de forma otimista
        que células DESCONHECIDAS são transitáveis.

        Args:
            start (tuple[int, int]): Coordenada de origem.
            goal (tuple[int, int]): Coordenada de destino.

        Returns:
            list[str]: Sequência ordenada de ações para alcançar o destino.
            Retorna lista vazia se for inalcançável.
        """
        parent = {start: (None, None)}
        queue  = deque([start])

        while queue:
            current = queue.popleft()
            if current == goal:
                return self._reconstruct(parent, start, goal)

            for action, (dr, dc) in _MOVE_DELTA.items():
                nb = (current[0] + dr, current[1] + dc)
                if nb not in parent and self._map.get(nb, UNKNOWN) != OBSTACLE:
                    parent[nb] = (current, action)
                    queue.append(nb)

        return []

    @staticmethod
    def _reconstruct(
        parent: dict,
        start:  tuple[int, int],
        goal:   tuple[int, int],
    ) -> list[str]:
        """
        Reconstrói o vetor de navegação processando a árvore do BFS de trás para frente.

        Args:
            parent (dict): Mapeamento de histórico (célula_atual -> célula_anterior, ação).
            start (tuple[int, int]): Ponto inicial da busca.
            goal (tuple[int, int]): Ponto final (alvo).

        Returns:
            list[str]: Caminho cronológico final de ações.
        """
        path, current = [], goal
        while current != start:
            current, action = parent[current]
            path.append(action)
        path.reverse()
        return path

    def reset(self) -> None:
        """Redefine completamente o histórico, mapa, coordenadas e planos de rota do agente."""
        super().reset()
        self._pos = (0, 0)
        self._map = {(0, 0): CLEAN}
        self._path = []
        self._last_action = None

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ── Color map ────────────────────────────────────────────────────────────────
CELL_COLORS = {
    CLEAN:    '#F5F5F0',
    DIRTY:    '#C8A96E',
    OBSTACLE: '#3D3D3A',
}
AGENT_COLOR    = '#E05A2B'
TRAIL_COLOR    = '#A8C5E8'
GRID_LINE_COLOR = '#DDDDD8'


def build_frame_history(env: 'Environment', agent: 'Agent', max_steps: int = 200) -> list[dict]:
    """
    Run a single simulation episode and record a snapshot after every action.

    Each snapshot contains:
        - grid       : copy of the environment grid at that moment
        - agent_pos  : (row, col) of the agent
        - action     : action executed to reach this state
        - step       : step index
        - score_m1   : cumulative Metric 1 score
        - score_m2   : cumulative Metric 2 score
    """
    history = []
    score_m1, score_m2 = 0, 0
    prev_dirty = int(np.sum(env.grid == DIRTY))

    history.append({
        'grid':      env.grid.copy(),
        'agent_pos': env.agent_pos,
        'action':    'START',
        'step':      0,
        'score_m1':  0,
        'score_m2':  0,
    })

    for step in range(1, max_steps + 1):
        percept = env.get_percept()
        action  = agent.decide(percept)
        agent.record(action)

        was_dirty = env.grid[env.agent_pos] == DIRTY
        env.apply_action(action)

        cleaned = was_dirty and action == 'SUCK'
        moved   = action in DELTA and not env.last_bump

        if cleaned:
            score_m1 += 1
            score_m2 += 1
        if moved:
            score_m2 -= 1

        history.append({
            'grid':      env.grid.copy(),
            'agent_pos': env.agent_pos,
            'action':    action,
            'step':      step,
            'score_m1':  score_m1,
            'score_m2':  score_m2,
        })

        if env.is_done():
            break

    return history


def animate_episode(
    history:    list[dict],
    title:      str = 'Agent',
    interval:   int = 120,
    trail_len:  int = 12,
) -> HTML:
    """
    Convert a frame history into an interactive HTML5 video player.

    Parameters
    ----------
    history   : output of build_frame_history()
    title     : label shown in the plot title
    interval  : milliseconds between frames
    trail_len : number of past positions to highlight as a movement trail
    """
    rows, cols = history[0]['grid'].shape
    fig, ax    = plt.subplots(figsize=(max(5, cols * 0.7), max(5, rows * 0.7)))
    fig.patch.set_facecolor('#FAFAF8')
    ax.set_facecolor('#FAFAF8')

    for x in range(cols + 1):
        ax.axvline(x, color=GRID_LINE_COLOR, linewidth=0.5, zorder=1)
    for y in range(rows + 1):
        ax.axhline(y, color=GRID_LINE_COLOR, linewidth=0.5, zorder=1)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect('equal')
    ax.axis('off')

    patches = {}
    for r in range(rows):
        for c in range(cols):
            rect = mpatches.FancyBboxPatch(
                (c + 0.05, rows - r - 0.95),
                0.9, 0.9,
                boxstyle='round,pad=0.05',
                linewidth=0,
                color=CELL_COLORS[CLEAN],
                zorder=2,
            )
            ax.add_patch(rect)
            patches[(r, c)] = rect

    agent_circle = plt.Circle((0, 0), 0.28, color=AGENT_COLOR, zorder=5)
    ax.add_patch(agent_circle)

    trail_circles = [
        plt.Circle((0, 0), 0.12,
                   color=TRAIL_COLOR,
                   alpha=0,
                   zorder=3)
        for _ in range(trail_len)
    ]
    for tc in trail_circles:
        ax.add_patch(tc)

    title_text = ax.set_title('', fontsize=11, pad=8, color='#2C2C2A')
    info_text  = ax.text(
        0.01, -0.03, '',
        transform=ax.transAxes,
        fontsize=9,
        color='#5F5E5A',
        va='top',
    )

    legend_elements = [
        mpatches.Patch(color=CELL_COLORS[DIRTY],    label='Dirty'),
        mpatches.Patch(color=CELL_COLORS[CLEAN],    label='Clean'),
        mpatches.Patch(color=CELL_COLORS[OBSTACLE], label='Obstacle'),
        mpatches.Patch(color=AGENT_COLOR,           label='Agent'),
        mpatches.Patch(color=TRAIL_COLOR,           label='Trail'),
    ]
    ax.legend(
        handles=legend_elements,
        loc='upper right',
        bbox_to_anchor=(1.18, 1.02),
        fontsize=8,
        framealpha=0.9,
        edgecolor=GRID_LINE_COLOR,
    )

    def update(frame_idx: int):
        frame = history[frame_idx]
        grid  = frame['grid']
        ar, ac = frame['agent_pos']
        for r in range(rows):
            for c in range(cols):
                patches[(r, c)].set_color(CELL_COLORS[grid[r, c]])

        agent_circle.center = (ac + 0.5, rows - ar - 0.5)
        trail_start = max(0, frame_idx - trail_len)
        for i, tc in enumerate(trail_circles):
            hist_idx = trail_start + i
            if hist_idx < frame_idx:
                tr, tc_col = history[hist_idx]['agent_pos']
                tc.center  = (tc_col + 0.5, rows - tr - 0.5)
                tc.set_alpha(0.15 + 0.55 * (i / trail_len))
            else:
                tc.set_alpha(0)

        title_text.set_text(
            f'{title}  —  step {frame["step"]} / {len(history) - 1}'
            f'   [{frame["action"]}]'
        )
        info_text.set_text(
            f'M1 (coverage): {frame["score_m1"]:+d}     '
            f'M2 (efficiency): {frame["score_m2"]:+d}'
        )

        return list(patches.values()) + [agent_circle] + trail_circles + [title_text, info_text]

    anim = FuncAnimation(
        fig,
        update,
        frames=len(history),
        interval=interval,
        blit=True,
    )

    plt.tight_layout()
    plt.close(fig)
    return HTML(anim.to_jshtml())

## Protocolo Experimental

### Motivação

Avaliar um agente em um único ambiente é metodologicamente insuficiente. Um agente reativo pode, por acaso, performar bem em uma sala pequena e sem obstáculos; um agente baseado em modelos pode ser penalizado em um ambiente tão aberto que a ausência de paredes elimina sua vantagem estrutural. Para que a comparação seja válida, é necessário medir o desempenho médio sobre uma distribuição de ambientes.

### Delineamento

O experimento consiste em $N = 50$ iterações independentes. Em cada iteração:

1. Um ambiente é gerado com parâmetros aleatorizados dentro dos intervalos definidos abaixo.
2. O **Agente Reativo Simples** é instanciado e executado nesse ambiente até limpeza total ou limite de passos.
3. O ambiente é **resetado ao estado inicial exato** via `env.clone()`.
4. O **Agente Baseado em Modelos** é instanciado e executado no mesmo ambiente.
5. Ambas as pontuações ($M_1$ e $M_2$) são registradas para os dois agentes.

O controle experimental crítico está no passo 3: os dois agentes enfrentam **geografias idênticas**. Isso elimina a variância ambiental como fator de confusão e garante que qualquer diferença de desempenho seja atribuível exclusivamente à arquitetura do agente.

### Parâmetros de Geração de Ambientes

| Parâmetro | Intervalo |
|---|---|
| Dimensões da grade | $6 \times 6$ a $12 \times 12$ |
| Probabilidade de sujeira por célula | $20\%$ a $40\%$ |
| Probabilidade de obstáculo por célula | $10\%$ a $25\%$ |
| Limite máximo de passos | $300$ |

### Resultado Esperado

Sob $M_1$, ambos os agentes devem apresentar desempenho moderado — o agente reativo eventualmente cobre o espaço por deriva aleatória. Sob $M_2$, espera-se uma separação significativa: o agente reativo acumula custo de movimento sem contrapartida de cobertura, enquanto o agente baseado em modelos, ao evitar revisitas, mantém saldo positivo por mais tempo. A magnitude dessa separação constitui a medida quantitativa do valor da memória neste domínio.

In [ ]:
import pandas as pd
import random
import copy

N_RUNS      = 50
MAX_STEPS   = 300
GRID_MIN    = 6
GRID_MAX    = 12
DIRT_MIN    = 0.20
DIRT_MAX    = 0.40
OBS_MIN     = 0.10
OBS_MAX     = 0.25

MASTER_SEED = 42
rng         = random.Random(MASTER_SEED)


def run_episode(env: 'Environment', agent: 'Agent', max_steps: int) -> dict:
    """
    Execute one episode and return final scores.

    Returns a dict with keys: score_m1, score_m2, steps, cleaned, total_dirty.
    """
    score_m1, score_m2 = 0, 0
    steps_taken        = 0
    cleaned_cells      = 0

    for _ in range(max_steps):
        percept  = env.get_percept()
        action   = agent.decide(percept)
        agent.record(action)

        was_dirty = env.grid[env.agent_pos] == DIRTY
        env.apply_action(action)

        if was_dirty and action == 'SUCK':
            score_m1    += 1
            score_m2    += 1
            cleaned_cells += 1

        if action in DELTA and not env.last_bump:
            score_m2    -= 1

        steps_taken += 1

        if env.is_done():
            break

    return {
        'score_m1':    score_m1,
        'score_m2':    score_m2,
        'steps':       steps_taken,
        'cleaned':     cleaned_cells,
        'total_dirty': env.initial_dirty,
    }


records = []

for run_id in range(N_RUNS):
    env_seed = rng.randint(0, 10_000)
    rows = rng.randint(GRID_MIN, GRID_MAX)
    cols = rng.randint(GRID_MIN, GRID_MAX)
    dirt_prob = rng.uniform(DIRT_MIN, DIRT_MAX)
    obs_prob = rng.uniform(OBS_MIN,  OBS_MAX)

    env_template = Environment(
        rows=rows,
        cols=cols,
        dirt_prob=dirt_prob,
        obstacle_prob=obs_prob,
        seed=env_seed,
    )

    env_reactive = copy.deepcopy(env_template)
    agent_reactive = SimpleReactiveAgent(seed=env_seed)
    result_reactive = run_episode(env_reactive, agent_reactive, MAX_STEPS)

    records.append({
        'run':         run_id,
        'agent':       'Reativo Simples',
        'rows':        rows,
        'cols':        cols,
        'dirt_prob':   round(dirt_prob, 3),
        'obs_prob':    round(obs_prob,  3),
        **result_reactive,
    })

    env_model = copy.deepcopy(env_template)
    agent_model = ModelBasedAgent()
    result_model = run_episode(env_model, agent_model, MAX_STEPS)

    records.append({
        'run':         run_id,
        'agent':       'Baseado em Modelos',
        'rows':        rows,
        'cols':        cols,
        'dirt_prob':   round(dirt_prob, 3),
        'obs_prob':    round(obs_prob,  3),
        **result_model,
    })

# ── Assemble DataFrame ────────────────────────────────────────────────────────

df = pd.DataFrame(records)
df['coverage_pct'] = (df['cleaned'] / df['total_dirty'].replace(0, 1) * 100).round(1)

# ── Summary statistics ────────────────────────────────────────────────────────

summary = (
    df
    .groupby('agent')[['score_m1', 'score_m2', 'steps', 'coverage_pct']]
    .agg(['mean', 'std'])
    .round(2)
)

print(f"Experiment complete — {N_RUNS} runs × 2 agents = {len(df)} episodes\n")
print(summary)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Style ─────────────────────────────────────────────────────────────────────

sns.set_theme(style='whitegrid', font_scale=1.05)

PALETTE = {
    'Reativo Simples':   '#C8A96E',
    'Baseado em Modelos': '#3B7FC4',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
fig.patch.set_facecolor('#FAFAF8')
fig.suptitle(
    'Comparação de Desempenho — Agente Reativo vs. Agente Baseado em Modelos',
    fontsize=13, fontweight='500', color='#2C2C2A', y=1.02,
)

# ── Shared helpers ────────────────────────────────────────────────────────────

def style_ax(ax, title, ylabel, xlabel=''):
    ax.set_title(title, fontsize=11, color='#2C2C2A', pad=10)
    ax.set_ylabel(ylabel, fontsize=9, color='#5F5E5A')
    ax.set_xlabel(xlabel, fontsize=9, color='#5F5E5A')
    ax.tick_params(colors='#5F5E5A', labelsize=9)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.set_facecolor('#FAFAF8')
    for spine in ax.spines.values():
        spine.set_visible(False)


# ── Plot 1: Global average score — M1 and M2 side by side ────────────────────

ax1 = axes[0]

means = df.groupby('agent')[['score_m1', 'score_m2']].mean().reset_index()
means_melted = means.melt(id_vars='agent', var_name='metric', value_name='mean_score')
means_melted['metric'] = means_melted['metric'].map({
    'score_m1': 'M1 — Cobertura',
    'score_m2': 'M2 — Eficiência',
})

sns.barplot(
    data=means_melted,
    x='metric', y='mean_score',
    hue='agent',
    palette=PALETTE,
    edgecolor='none',
    alpha=0.88,
    ax=ax1,
)

# Annotate bar values
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.1f', label_type='edge', fontsize=8.5,
                  color='#3D3D3A', padding=3)

ax1.axhline(0, color='#3D3D3A', linewidth=0.8, linestyle='-')
ax1.legend(title='', fontsize=8.5, framealpha=0)
style_ax(ax1, 'Pontuação Média Global', 'Pontuação média')


# ── Plot 2: Per-run M2 score distribution (violin) ───────────────────────────

ax2 = axes[1]

sns.violinplot(
    data=df,
    x='agent', y='score_m2',
    palette=PALETTE,
    inner='quartile',
    linewidth=0.8,
    alpha=0.82,
    ax=ax2,
)

ax2.axhline(0, color='#3D3D3A', linewidth=0.8, linestyle='--', alpha=0.6)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=12, ha='right')
style_ax(ax2, 'Distribuição de M2 por Corrida', 'M2 — Eficiência')


# ── Plot 3: Steps taken vs. dirty cells cleaned (scatter) ────────────────────

ax3 = axes[2]

for agent_name, group in df.groupby('agent'):
    ax3.scatter(
        group['steps'],
        group['cleaned'],
        label=agent_name,
        color=PALETTE[agent_name],
        alpha=0.65,
        s=40,
        edgecolors='none',
    )

    # Trend line
    z = np.polyfit(group['steps'], group['cleaned'], 1)
    p = np.poly1d(z)
    xs = np.linspace(group['steps'].min(), group['steps'].max(), 100)
    ax3.plot(xs, p(xs), color=PALETTE[agent_name], linewidth=1.2,
             linestyle='--', alpha=0.55)

ax3.legend(title='', fontsize=8.5, framealpha=0)
style_ax(
    ax3,
    'Passos Realizados vs. Células Limpas',
    'Células limpas',
    'Passos realizados',
)


# ── Render ────────────────────────────────────────────────────────────────────

plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

### Resultados Observados

A execução do experimento sobre 50 ambientes independentes produz um padrão consistente com a hipótese formulada na Célula 1. Os dados revelam três fenômenos distintos:

**Sob M1 — Cobertura Pura**, o Agente Reativo apresenta desempenho não negligenciável. Em ambientes suficientemente pequenos, a deriva aleatória garante cobertura razoável dentro do limite de 300 passos. A diferença entre os agentes existe, mas é moderada — o agente reativo *eventualmente* chega às células sujas.

**Sob M2 — Eficiência**, a separação é pronunciada. O Agente Reativo acumula custo de movimento de forma descontrolada: sem memória de paredes já mapeadas, repete colisões; sem registro de células já limpas, revisita posições inúteis. Cada passo desperdiçado desconta um ponto que ele não recuperará. Em ambientes com alta densidade de obstáculos, o saldo de M2 do agente reativo frequentemente torna-se negativo — ele moveu-se mais do que limpou.

**No gráfico de dispersão** (passos vs. células limpas), a diferença estrutural torna-se visível geometricamente: os pontos do Agente Reativo dispersam-se horizontalmente — muitos passos, cobertura variável — enquanto os pontos do Agente Baseado em Modelos concentram-se na região de alta cobertura com menor número de passos, refletindo trajetórias mais diretas.

### Interpretação

O resultado não é surpreendente, mas é instrutivo. O Agente Reativo falha não por falta de capacidade sensorial — ambos os agentes recebem exatamente a mesma percepção — mas por incapacidade de *acumular evidência*. Cada passo é tratado como se fosse o primeiro. A parede que ele encontrou no passo 12 é desconhecida no passo 47.

O Agente Baseado em Modelos converte percepções locais e sequenciais em conhecimento global e persistente. O mapa interno não é um luxo — é a condição mínima para que BFS produza caminhos válidos. Sem ele, não há planejamento; com ele, o agente nunca precisa descobrir a mesma parede duas vezes.

### Limitação e Extensão

Este experimento opera em ambientes estáticos: obstáculos e sujeira não mudam durante o episódio. Em ambientes dinâmicos — sujeira que reaparece, obstáculos móveis — o mapa interno do agente baseado em modelos tornar-se-ia progressivamente desatualizado, e a vantagem observada aqui se reduziria. Uma extensão natural seria introduzir um **mecanismo de expiração de memória**, forçando o agente a reexplorar regiões não visitadas há mais de $k$ passos. Esse compromisso entre memória e reexploração constitui um problema central em robótica autônoma e planejamento sob incerteza.

### Conclusão

> Memória não amplia os sensores do agente — ela amplifica o valor de cada observação, distribuindo seu impacto ao longo de toda a trajetória futura. A eficiência não é uma propriedade do ambiente; é uma consequência da arquitetura.